In [0]:
# Databricks notebook source
# GOLD LAYER: Driver Championship Standings
# Purpose: Cumulative points, championship position, win/podium counts
# Source: f1_data.silver.fact_results (race sessions only)

from pyspark.sql.functions import (
    col, sum as spark_sum, count, when, lit,
    round as spark_round, rank, dense_rank,
    row_number, current_timestamp, max as spark_max,
    avg, first, last
)
from pyspark.sql.window import Window

# ============ 1. READ FROM SILVER ============
fact = spark.read.table("f1_data.silver.fact_results")

# Gold layer uses RACE sessions only for championship standings
# Sprint results may also count depending on your dataset
race_results = fact.filter(
    col("session_type").isin("Race", "Sprint")
)

print(f"Race/Sprint results loaded: {race_results.count()}")

# ============ 2. AGGREGATE SEASON STATS PER DRIVER ============
# Define a season window that takes the driver number and the year
season_window = Window.partitionBy("driver_number", "year")

# Define a cumulative window that takes the driver number and the year, and orders by date and look from the very first available row in that 
# bucket (unboundedPreceding) up to the current row it is processing (currentRow)
cumulative_window = Window.partitionBy("driver_number", "year") \
    .orderBy("session_date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Step 1: Cumulative stats per race (for running standings)
cumulative_stats = race_results \
    .withColumn("cumulative_points",
        spark_sum("points").over(cumulative_window) # This column contains the gained points accumulated up to the current row
    ) \
    .withColumn("cumulative_wins",
        spark_sum(when(col("position") == 1, 1).otherwise(0)).over(cumulative_window) # This column contains the attained wins up to the current row
    ) \
    .withColumn("cumulative_podiums",
        spark_sum(when(col("position") <= 3, 1).otherwise(0)).over(cumulative_window) # This column contains the attained podiums up to the current row
    ) \
    .withColumn("cumulative_dnfs",
        spark_sum(when(col("dnf") == True, 1).otherwise(0)).over(cumulative_window) # This column contains the attained dnfs up to the current row
    ) \
    .withColumn("cumulative_races",
        count("session_key").over(cumulative_window) # This column contains the races participated up to the current row
    )

# Step 2: Championship position per race weekend (rolling rank)
# Define a position window that takes the year and session date, and orders by cumulative points
position_window = Window.partitionBy("year", "session_date") \
    .orderBy(col("cumulative_points").desc())

cumulative_with_rank = cumulative_stats \
    .withColumn("championship_position_after_race",
        dense_rank().over(position_window)
    ) \
    .withColumn("points_behind_leader",
        spark_max("cumulative_points").over(
            Window.partitionBy("year", "session_date")
        ) - col("cumulative_points")
    )

# Step 3: Final season standings (latest state per driver per year)
latest_window = Window.partitionBy("driver_number", "year") \
    .orderBy(col("session_date").desc())

final_standings = cumulative_with_rank \
    .withColumn("row_num", row_number().over(latest_window)) \
    .filter(col("row_num") == 1) \
    .drop("row_num")

# ============ 3. BUILD FINAL STANDINGS TABLE ============
championship_window = Window.partitionBy("year") \
    .orderBy(col("cumulative_points").desc())

driver_standings = final_standings \
    .withColumn("championship_position",
        dense_rank().over(championship_window) # DENSE_RANK provides consecutive ranks without any gaps while rank skips numbers after a tie
    ) \
    .withColumn("points_behind_leader",
        spark_max("cumulative_points").over(Window.partitionBy("year"))
        - col("cumulative_points") # With the leader having the highest points, determine the deficit per row
    ) \
    .withColumn("win_rate",
        spark_round(col("cumulative_wins") / col("cumulative_races") * 100, 2) # Win rate = wins / races
    ) \
    .withColumn("podium_rate",
        spark_round(col("cumulative_podiums") / col("cumulative_races") * 100, 2) # Podium rate = podiums / races
    ) \
    .withColumn("dnf_rate",
        spark_round(col("cumulative_dnfs") / col("cumulative_races") * 100, 2) # DNF rate = dnfs / races
    ) \
    .withColumn("avg_points_per_race",
        spark_round(col("cumulative_points") / col("cumulative_races"), 2) # Average points per race = total points / races
    ) \
    .withColumn("gold_processed_at", current_timestamp()) \
    .withColumn("gold_source", lit("f1_data.silver.fact_results"))

# ============ 4. SELECT FINAL COLUMNS ============
driver_standings_final = driver_standings.select(
    # Identity
    "driver_number",
    "year",
    "driver_name",
    "driver_acronym",
    "team_name",
    "team_colour",
    "driver_country_code",

    # Standing
    "championship_position",
    "points_behind_leader",

    # Totals
    col("cumulative_points").alias("total_points"),
    col("cumulative_races").alias("races_entered"),
    col("cumulative_wins").alias("total_wins"),
    col("cumulative_podiums").alias("total_podiums"),
    col("cumulative_dnfs").alias("total_dnfs"),

    # Rates
    "win_rate",
    "podium_rate",
    "dnf_rate",
    "avg_points_per_race",

    # Last race context
    col("session_date").alias("last_race_date"),
    col("meeting_name").alias("last_race_name"),
    col("position").alias("last_race_position"),
    col("points").alias("last_race_points"),

    # Metadata
    "gold_processed_at",
    "gold_source"
)

# ============ 5. WRITE TO GOLD ============
driver_standings_final.write \
    .mode("overwrite") \
    .format("delta") \
    .partitionBy("year") \
    .saveAsTable("f1_data.gold.driver_standings")

print(f"✓ gold.driver_standings written: {driver_standings_final.count()} records")

# ============ 6. QUICK VALIDATION ============
print("\n🏆 CURRENT CHAMPIONSHIP STANDINGS:")
spark.sql("""
    SELECT 
        championship_position,
        driver_name,
        team_name,
        total_points,
        total_wins,
        total_podiums,
        avg_points_per_race
    FROM f1_data.gold.driver_standings
    WHERE year = 2024
    ORDER BY championship_position
""").show(20, truncate=False)

Race/Sprint results loaded: 419
✓ gold.driver_standings written: 23 records

🏆 CURRENT CHAMPIONSHIP STANDINGS:
+---------------------+----------------+---------------+------------+----------+-------------+-------------------+
|championship_position|driver_name     |team_name      |total_points|total_wins|total_podiums|avg_points_per_race|
+---------------------+----------------+---------------+------------+----------+-------------+-------------------+
|1                    |Max Verstappen  |Red Bull Racing|331.0       |10        |14           |15.76              |
|2                    |Lando Norris    |McLaren        |279.0       |3         |12           |13.29              |
|3                    |Charles Leclerc |Ferrari        |245.0       |2         |10           |11.67              |
|4                    |Oscar Piastri   |McLaren        |237.0       |2         |8            |11.29              |
|5                    |Carlos Sainz    |Ferrari        |190.0       |1         |5   

In [0]:
display(final_standings)

driver_number,meeting_key,session_key,driver_name,driver_first_name,driver_last_name,driver_acronym,team_name,team_colour,driver_country_code,session_name,session_type,session_date,year,meeting_name,meeting_official_name,circuit_short_name,circuit_type,race_country,race_location,position,points,duration,number_of_laps,dnf,dns,dsq,gap_to_leader_raw,gap_to_leader_seconds,gap_type,finished_race,on_podium,points_per_lap,driver_found_in_dim,silver_source,silver_processed_at,data_quality_flag,cumulative_points,cumulative_wins,cumulative_podiums,cumulative_dnfs,cumulative_races,championship_position_after_race,points_behind_leader
1,1246,9606,Max Verstappen,Max,Verstappen,VER,Red Bull Racing,3671C6,NED,Race,Race,2024-09-22T12:00:00.000Z,2024,Singapore Grand Prix,FORMULA 1 SINGAPORE AIRLINES SINGAPORE GRAND PRIX 2024,Singapore,TEMPORARY - STREET,Singapore,Marina Bay,2.0,18.0,6073.516,62,false,false,false,20.945,20.945,TIME_GAP,true,true,0.2903,true,f1_data.bronze.results,2026-07-03T09:43:41.959Z,PASSED,331.0,10,14,1,21,1,0.0
2,1243,9582,Logan Sargeant,Logan,Sargeant,SAR,Williams,64C4FF,USA,Race,Race,2024-08-25T13:00:00.000Z,2024,Dutch Grand Prix,FORMULA 1 HEINEKEN DUTCH GRAND PRIX 2024,Zandvoort,PERMANENT,Netherlands,Zandvoort,16.0,0.0,null,71,false,false,false,+1 LAP,null,LAPPED,true,false,0.0,true,f1_data.bronze.results,2026-07-03T09:43:41.959Z,PASSED,0.0,0,0,2,17,16,295.0
3,1246,9606,Daniel Ricciardo,Daniel,Ricciardo,RIC,RB,6692FF,AUS,Race,Race,2024-09-22T12:00:00.000Z,2024,Singapore Grand Prix,FORMULA 1 SINGAPORE AIRLINES SINGAPORE GRAND PRIX 2024,Singapore,TEMPORARY - STREET,Singapore,Marina Bay,18.0,0.0,null,61,false,false,false,+1 LAP,null,LAPPED,true,false,0.0,true,f1_data.bronze.results,2026-07-03T09:43:41.959Z,PASSED,12.0,0,0,2,21,12,319.0
4,1246,9606,Lando Norris,Lando,Norris,NOR,McLaren,FF8000,GBR,Race,Race,2024-09-22T12:00:00.000Z,2024,Singapore Grand Prix,FORMULA 1 SINGAPORE AIRLINES SINGAPORE GRAND PRIX 2024,Singapore,TEMPORARY - STREET,Singapore,Marina Bay,1.0,25.0,6052.571,62,false,false,false,0,null,TIME_GAP,true,true,0.4032,true,f1_data.bronze.results,2026-07-03T09:43:41.959Z,PASSED,279.0,3,12,1,21,2,52.0
10,1246,9606,Pierre Gasly,Pierre,Gasly,GAS,Alpine,0093CC,FRA,Race,Race,2024-09-22T12:00:00.000Z,2024,Singapore Grand Prix,FORMULA 1 SINGAPORE AIRLINES SINGAPORE GRAND PRIX 2024,Singapore,TEMPORARY - STREET,Singapore,Marina Bay,17.0,0.0,null,61,false,false,false,+1 LAP,null,LAPPED,true,false,0.0,true,f1_data.bronze.results,2026-07-03T09:43:41.959Z,PASSED,8.0,0,0,2,21,13,323.0
11,1246,9606,Sergio Perez,Sergio,Perez,PER,Red Bull Racing,3671C6,MEX,Race,Race,2024-09-22T12:00:00.000Z,2024,Singapore Grand Prix,FORMULA 1 SINGAPORE AIRLINES SINGAPORE GRAND PRIX 2024,Singapore,TEMPORARY - STREET,Singapore,Marina Bay,10.0,1.0,null,61,false,false,false,+1 LAP,null,LAPPED,true,false,0.0164,true,f1_data.bronze.results,2026-07-03T09:43:41.959Z,PASSED,144.0,0,6,3,21,8,187.0
14,1246,9606,Fernando Alonso,Fernando,Alonso,ALO,Aston Martin,229971,ESP,Race,Race,2024-09-22T12:00:00.000Z,2024,Singapore Grand Prix,FORMULA 1 SINGAPORE AIRLINES SINGAPORE GRAND PRIX 2024,Singapore,TEMPORARY - STREET,Singapore,Marina Bay,8.0,4.0,null,61,false,false,false,+1 LAP,null,LAPPED,true,false,0.0656,true,f1_data.bronze.results,2026-07-03T09:43:41.959Z,PASSED,62.0,0,0,1,21,9,269.0
16,1246,9606,Charles Leclerc,Charles,Leclerc,LEC,Ferrari,E80020,MON,Race,Race,2024-09-22T12:00:00.000Z,2024,Singapore Grand Prix,FORMULA 1 SINGAPORE AIRLINES SINGAPORE GRAND PRIX 2024,Singapore,TEMPORARY - STREET,Singapore,Marina Bay,5.0,10.0,6115.001,62,false,false,false,62.43,62.43,TIME_GAP,true,false,0.1613,true,f1_data.bronze.results,2026-07-03T09:43:41.959Z,PASSED,245.0,2,10,1,21,3,86.0
18,1246,9606,Lance Stroll,Lance,Stroll,STR,Aston Martin,229971,CAN,Race,Race,2024-09-22T12:00:00.000Z,2024,Singapore Grand Prix,FORMULA 1 SINGAPORE AIRLINES SINGAPORE GRAND PRIX 2024,Singapore,TEMPORARY - STREET,Singapore,Marina Bay,14.0,0.0,null,61,false,false,false,+1 LAP,null,LAPPED,tru